In [ ]:
def _add_system_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from AEMO DISPATCHREGIONSUM system-level columns:
    available generation, net interchange, and demand forecast.
    These are strong predictors of supply tightness and price spikes.
    """
    df = df.copy()
    ag = df["avail_gen"]
    ic = df["interchange"]
    dc = df["demand_forecast"]
    dm = df["demand"]

    # Reserve margin: how much headroom between available gen and demand
    df["reserve_margin"]     = ((ag - dm) / (dm + 1)).clip(-2, 10).astype(np.float32)
    df["reserve_margin_pct"] = (ag / (dm + 1)).clip(0, 5).astype(np.float32)

    # Demand forecast vs actual demand (indicates demand surprise)
    df["demand_fcst_error"]  = (dc - dm).astype(np.float32)

    # Lags for available generation
    for lag in [1, 2, 4, 48, 96, 336]:
        df[f"avail_gen_lag_{lag}"] = ag.shift(lag).astype(np.float32)

    # Lags for net interchange (positive = importing into NSW)
    for lag in [1, 2, 4, 48, 96, 336]:
        df[f"interchange_lag_{lag}"] = ic.shift(lag).astype(np.float32)
    df["interchange_rmean_48"]  = ic.rolling(48).mean().astype(np.float32)
    df["interchange_rmean_336"] = ic.rolling(336).mean().astype(np.float32)

    # Demand forecast lags
    for lag in [1, 2, 48]:
        df[f"demand_fcst_lag_{lag}"] = dc.shift(lag).astype(np.float32)

    # Reserve margin rolling stats (tightening/loosening supply)
    rm = df["reserve_margin"]
    df["reserve_rmean_48"]  = rm.rolling(48).mean().astype(np.float32)
    df["reserve_rmin_48"]   = rm.rolling(48).min().astype(np.float32)

    # Dispatched generation features (from DISPATCHABLEGENERATION column)
    if "dispatch_gen" in df.columns:
        dg = df["dispatch_gen"]
        # Thermal utilisation: what fraction of available capacity is dispatched
        df["thermal_util"]       = (dg / (ag + 1)).clip(0, 1.5).astype(np.float32)
        # Spare thermal headroom (MW) — low values precede price spikes
        df["thermal_surplus_mw"] = (ag - dg).clip(-2000, 8000).astype(np.float32)
        for lag in [1, 2, 4, 48, 96, 336]:
            df[f"dispatch_gen_lag_{lag}"] = dg.shift(lag).astype(np.float32)
        df["dispatch_gen_chg_4h"]   = dg.diff(8).astype(np.float32)    # 4h change
        df["dispatch_gen_chg_24h"]  = dg.diff(48).astype(np.float32)   # 24h change
        df["dispatch_gen_rmin_48"]  = dg.rolling(48).min().astype(np.float32)
        df["dispatch_gen_rmean_48"] = dg.rolling(48).mean().astype(np.float32)
        df["thermal_surplus_rmin_48"] = df["thermal_surplus_mw"].rolling(48).min().astype(np.float32)

    return df

In [ ]:
def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:

    _NSW_HOLIDAYS = holidays.Australia(state="NSW", years=range(2018, 2025))
    
    idx = df.index
    df = df.copy()

    # Raw calendar components
    df["hour"]        = idx.hour.astype(np.int8)
    df["dayofweek"]   = idx.dayofweek.astype(np.int8)
    df["month"]       = idx.month.astype(np.int8)
    df["dayofyear"]   = idx.day_of_year.astype(np.int16)

    # Cyclical (sin/cos) encodings so the model sees periodicity
    df["hour_sin"]    = np.sin(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["hour_cos"]    = np.cos(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)
    df["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)

    # Binary flags
    df["is_weekend"]  = (idx.dayofweek >= 5).astype(np.float32)
    df["is_holiday"]  = np.array(
        [d.date() in _NSW_HOLIDAYS for d in idx], dtype=np.float32
    )
    # Peak (17–20 h) and off-peak (< 7 h or ≥ 21 h) periods
    df["is_peak"]     = ((idx.hour >= 17) & (idx.hour <= 20)).astype(np.float32)
    df["is_shoulder"] = ((idx.hour >= 7)  & (idx.hour < 17)).astype(np.float32)
    df["is_off_peak"] = ((idx.hour < 7)   | (idx.hour >= 21)).astype(np.float32)

    # Combined weekend/holiday flag — demand behaviour is quite different
    df["is_offday"]   = np.maximum(df["is_weekend"], df["is_holiday"]).astype(np.float32)

    return df


In [ ]:
def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Demand lags at key calendar anchors
    for lag in [48, 96, 336, 672]:
        df[f"demand_lag_{lag}"] = df["demand"].shift(lag).astype(np.float32)
    return df


In [ ]:
def _add_long_range_features(df: pd.DataFrame) -> pd.DataFrame:
    
    df   = df.copy()
    d    = df["demand"]
    BASE = 17_532

    # --- Annual demand lag ---
    df["demand_lag_annual"] = d.shift(BASE).astype(np.float32)

    # --- 2-week rolling statistics (not in main ROLLING_WINDOWS to keep loop tight) ---
    df["demand_rmean_2w"] = d.rolling(672, min_periods=336).mean().astype(np.float32)

    # 6-week rolling mean (seasonal trend)
    df["demand_rmean_6w"] = d.rolling(2016, min_periods=1008).mean().astype(np.float32)

    return df


In [ ]:
def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Rolling statistics computed on demand available at each timestamp."""
    df = df.copy()
    d = df["demand"]
    for w in [4, 24, 48, 96]:
        df[f"demand_rmean_{w}"] = d.rolling(w, min_periods=1).mean().astype(np.float32)
    return df


In [ ]:
def _add_spike_predictors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Supply-stress and generation features.
    All look-back only — zero leakage.
    """
    df = df.copy()
    ag = df["avail_gen"]
    dm = df["demand"]
    dc = df["demand_forecast"]

    # How loaded the system is (>1 means demand > available gen)
    supply_stress = dm / (ag + 1)
    df["supply_stress"]          = supply_stress.clip(0, 2).astype(np.float32)
    df["supply_stress_max_48"]   = supply_stress.rolling(48).max().astype(np.float32)
    df["supply_stress_max_96"]   = supply_stress.rolling(96).max().astype(np.float32)

    # Count of intervals in last 24h with very tight supply (>92% utilisation)
    df["tight_count_48"]  = (supply_stress > 0.92).rolling(48).sum().astype(np.float32)
    df["tight_count_336"] = (supply_stress > 0.92).rolling(336).sum().astype(np.float32)

    # Detects generator outages: large drops in available generation
    df["avail_gen_chg_24h"] = ag.pct_change(48).clip(-1, 1).astype(np.float32)
    df["avail_gen_rmin_48"] = ag.rolling(48).min().astype(np.float32)
    df["avail_gen_rmin_96"] = ag.rolling(96).min().astype(np.float32)

    # Short-term demand surprise: actual demand vs dispatch forecast
    df["demand_surprise"]     = (dm - dc).astype(np.float32)
    df["demand_surprise_abs"] = (dm - dc).abs().astype(np.float32)
    df["demand_surprise_rm48"] = (dm - dc).rolling(48).mean().astype(np.float32)

    # Supply headroom as percentage: (avail - demand) / demand
    df["supply_headroom_pct"] = ((ag - dm) / (dm + 1)).clip(-0.5, 1.0).astype(np.float32)

    # Forward tightness proxy: demand forecast vs available generation
    fwd_tight = (dc - ag).clip(-5000, 5000)
    df["fwd_tight_proxy"]      = fwd_tight.astype(np.float32)
    df["fwd_tight_proxy_lag48"] = fwd_tight.shift(48).astype(np.float32)

    return df
